In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
from tensorflow import keras
from tensorflow_model_optimization.python.core.keras.compat import keras #for Quantization Aware Training (QAT)
import tensorflow_model_optimization as tfmot #for Post Training Quantization (PTQ)
import keras_tuner as kt
from tensorflow.keras.layers import Conv1D, Dense, Flatten, Dropout, MaxPooling1D, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
import numpy as np
import scipy.io
import joblib
import numpy as np
import pandas as pd
from tabulate import tabulate

In [ ]:
# Check if GPU is Available
print("Num GPUs Available: ", tf.config.list_physical_devices('GPU'))
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    # Print information about each GPU
    for gpu in gpus:
        print("GPU:", gpu.name)
else:
    print("No GPU devices found.")

In [ ]:
# Configure GPU memory growth for training
if gpus:
    try:
        for gpu in gpus:
            # Allow TensorFlow to allocate memory as needed
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled")
    except RuntimeError as e:
        print("Memory growth setting failed:", e)

## Dataset Preparation

In [ ]:
import urllib.request
import tarfile
import sys

if not tf.io.gfile.exists("./speech_dataset"):
    remote = "https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
    local = "./speech_dataset.tar.gz"

    def progress(count, block_size, total_size):
        sys.stdout.write('\r>> Downloading %s %.1f%%' %('Speech dataset', float(count * block_size) / float(total_size) * 100.0))
        sys.stdout.flush()
    urllib.request.urlretrieve(remote, local, progress)

    tgz = tarfile.open(local)
    
    # Define the 10 classes we want to extract
    target_classes = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
    
    # Get the list of members to extract
    members = []
    total_members = 0
    for member in tgz.getmembers():
        # Check if the file belongs to one of our target classes
        parts = member.name.split('/')
        if len(parts) > 1 and parts[1] in target_classes:
            members.append(member)
            total_members += 1
    
    # Extract 
    print(f"\nExtracting 10 target classes: {', '.join(target_classes)}")
    for i, member in enumerate(members):
        tgz.extract(member, path="./speech_dataset")
        if i % 100 == 0 or i == total_members - 1:
            progress = (i + 1) / total_members * 100
            sys.stdout.write(f'\rExtracting files: {i+1}/{total_members} ({progress:.1f}%)')
            sys.stdout.flush()
    
    print("\nExtraction complete!")
    tgz.close()

In [ ]:
# Count the number of files in each class
target_classes = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
class_counts = {}
for target_class in target_classes:
    class_path = f"./speech_dataset/{target_class}"
    class_files = tf.io.gfile.listdir(class_path)
    class_counts[target_class] = len(class_files)
print("Class Counts:")
print(class_counts)
print("Total Files:", sum(class_counts.values()))


In [ ]:
# Load the dataset with 85-10-5 split
def load_data():
    X = []
    y = []
    global width, classes
    classes = {}
    width = 16000  # Most audio files are about 1 second at 16kHz
    
    # Load raw data and labels
    for i, target_class in enumerate(target_classes):
        classes[target_class] = i  # Create class mapping to integers
        class_path = f"./speech_dataset/{target_class}"
        class_files = tf.io.gfile.listdir(class_path)
        for file_name in class_files:
            file_path = f"{class_path}/{file_name}"
            with tf.io.gfile.GFile(file_path, 'rb') as file:
                wav_data = file.read()
                X.append(wav_data)
                y.append(target_class)
    
    # Split the dataset - 85% training, 15% for test+val
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
    # Further split test+val into test and validation (33% of 15% = 5% overall for test)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.33, random_state=42, stratify=y_temp)
    
    # Process the audio data for model input
    def process_audio(audio_list, labels):
        processed_audio = []
        processed_labels = []
        
        for audio, label in zip(audio_list, labels):
            # Decode WAV data
            wav_obj = tf.audio.decode_wav(tf.constant(audio))
            wav_data = wav_obj.audio.numpy().flatten()
            
            # Ensure all samples have the same length
            if len(wav_data) > width:
                print(f"Warning: Truncating file of length {len(wav_data)} to {width}")
                wav_data = wav_data[:width]
            elif len(wav_data) < width:
                print(f"Warning: Padding file of length {len(wav_data)} to {width}")
                wav_data = np.pad(wav_data, (0, width - len(wav_data)), 'constant')
            
            processed_audio.append(wav_data.reshape(width, 1))  # Shape for CNN input
            processed_labels.append(classes[label])  # Convert label to integer
        
        return np.array(processed_audio), np.array(processed_labels)
    
    # Process all datasets
    X_train, y_train = process_audio(X_train, y_train)
    X_val, y_val = process_audio(X_val, y_val)
    X_test, y_test = process_audio(X_test, y_test)
    
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = load_data()
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))
print("Shape of samples:", len(X_train[100]))

In [ ]:
# Plot some samples
def plot_samples(X, y, n=5):
    fig, axes = plt.subplots(1, n, figsize=(20, 4))
    for i in range(n):
        index = random.randint(0, len(X))
        wav_data = X[index]
        label = y[index]
        axes[i].plot(wav_data)
        axes[i].set_title(label)
    plt.show()
plot_samples(X_train, y_train)

## NN Evaluation Methods

In [ ]:
def evaluate_NN_models(model_list, X_test, y_test, bin_class=False):
    loss, accuracy, y_pred, precision, recall, f1_score, support, confusion_matrix_list = [], [], [], [], [], [], [], []
    for model in model_list:
        loss_value, accuracy_value = model.evaluate(X_test, y_test)
        loss.append(loss_value)
        accuracy.append(accuracy_value)
        if bin_class:
            y_pred_value = (model.predict(X_test) > 0.5).astype(int)
        else:
            y_pred_value = np.argmax(model.predict(X_test), axis=1)
        y_pred.append(y_pred_value)
        y_test_classes = y_test
        precision_value, recall_value, f1_score_value, support_value = precision_recall_fscore_support(
            y_test_classes, 
            y_pred_value, 
            average='binary' if bin_class else 'weighted',
            zero_division=0
        )
        precision.append(precision_value)
        recall.append(recall_value)
        f1_score.append(f1_score_value)
        support.append(support_value)
        confusion_matrix_value = confusion_matrix(y_test_classes, y_pred_value)
        confusion_matrix_list.append(confusion_matrix_value)
    return [accuracy, precision, recall, f1_score], confusion_matrix_list, y_pred

In [ ]:
def print_table(data,headers):
    table_data = list(zip(*data))
    table = tabulate(table_data, headers=headers, tablefmt='grid')
    print(table)

In [ ]:
def plot_confusion_matrix(ax, conf_matrix, title, cmap):
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap=cmap, cbar=False, ax=ax)
    ax.set_title(title)

In [ ]:
def draw_confusion_matrix(matrix,name):
    colormap=["Blues","Greens","Oranges"]
    fig, axes = plt.subplots(1, 1, figsize=(15, 5))
    for i in range(len(matrix)):
        plot_confusion_matrix(axes, matrix[i], "Confusion Matrix  ", colormap[i])
    plt.tight_layout()

In [ ]:
# Define early stopping callback
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=25,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

## Hyperparameters Optimization

In [ ]:
class QuantNetHyperModel(kt.HyperModel):
    def __init__(self, input_shape, num_classes):
        self.input_shape = input_shape
        self.num_classes = num_classes

    def build(self, hp):
        model = Sequential()
        
        filters_conv1 = hp.Int('conv1_filters', min_value=16, max_value=64, step=16)
        model.add(Conv1D(filters=filters_conv1, kernel_size=3, activation='relu', 
                        input_shape=self.input_shape))
        model.add(MaxPooling1D(pool_size=2))
        model.add(BatchNormalization(scale=False))

        filters_qconv1 = hp.Int('conv2_filters', min_value=4, max_value=16, step=4)
        model.add(Conv1D(filters_qconv1, kernel_size=3, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
        model.add(BatchNormalization(scale=False))
        model.add(Dropout(rate=hp.Float('dropout1', min_value=0.2, max_value=0.5)))

        filters_qconv2 = hp.Int('conv3_filters', min_value=4, max_value=16, step=4)
        
        model.add(Conv1D(filters_qconv2, kernel_size=3, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
        model.add(BatchNormalization(scale=False))
        model.add(Dropout(rate=hp.Float('dropout2', min_value=0.2, max_value=0.5)))

        model.add(Flatten())

        units_qdense = hp.Int('dense_units', min_value=4, max_value=16, step=4)
        model.add(Dense(units_qdense, activation='relu'))
        model.add(BatchNormalization(scale=False))
        model.add(Dense(self.num_classes, activation='softmax'))

        # Tune learning rate
        learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
        model.compile(
            optimizer=Adam(learning_rate=learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

In [ ]:
# Initialize the hypermodel
hypermodel = QuantNetHyperModel(input_shape=(width, 1), num_classes=len(classes))

# Create the tuner
tuner = kt.BayesianOptimization(
    hypermodel,
    objective='val_accuracy',
    max_trials=20,
    directory='bayesian_opt',
    project_name='sc_tuning'
)

# Search for the best hyperparameters
tuner.search(
    X_train,
    y_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

# Get the best hyperparameters
best_hps_list = tuner.get_best_hyperparameters(num_trials=3)
best_models_list = tuner.get_best_models(num_models=3)

In [ ]:
# Show the top 3 models
print("\nTop 3 Models Performance:")
print("-" * 50)

for idx, hps in enumerate(best_hps_list, 1):
    print(f"\nModel {idx}:")
    print("Hyperparameters:")
    for param, value in hps.values.items():
        print(f"{param}: {value}")
    print(f"Validation Accuracy: {best_models_list[idx-1].evaluate(X_val, y_val, verbose=0)[1]:.3f}")
        
# Let user choose the model
while True:
    try:
        choice = int(input("\nChoose a model (1, 2, or 3): "))
        if 1 <= choice <= 3:
            break
        print("Please enter 1, 2, or 3")
    except ValueError:
        print("Please enter a valid number")

# Use the chosen model
# model = best_models_list[choice - 1]
chosen_hps = best_hps_list[choice - 1]

print(f"\nSelected Model {choice}")
print("Final hyperparameters:")
for param, value in chosen_hps.values.items():
    print(f"{param}: {value}")

## Build BNN

Uncomment the following cell to train a model without hyperparameter tuning, otherwise train a model based on the hyperparameters found by the optimization process

In [ ]:
model = keras.models.Sequential()
model.add(keras.layers.Conv1D(filters=chosen_hps.get('conv1_filters'), kernel_size=3, activation='relu', 
                 input_shape=(width,1)))
model.add(keras.layers.MaxPooling1D(pool_size=2))
model.add(keras.layers.BatchNormalization(scale=False))

model.add(keras.layers.Conv1D(chosen_hps.get('conv2_filters'), kernel_size=3, activation='relu'))
model.add(keras.layers.MaxPooling1D(pool_size=2))
model.add(keras.layers.BatchNormalization(scale=False))
model.add(keras.layers.Dropout(rate=chosen_hps.get('dropout1')))

# Second QuantConv1D layer with tuned filters
model.add(keras.layers.Conv1D(chosen_hps.get('conv3_filters'), kernel_size=3,  activation='relu'))
model.add(keras.layers.MaxPooling1D(pool_size=2))
model.add(keras.layers.BatchNormalization(scale=False))
model.add(keras.layers.Dropout(rate=chosen_hps.get('dropout2')))

model.add(keras.layers.Flatten())

# QuantDense layer with tuned units
model.add(keras.layers.Dense(chosen_hps.get('dense_units'),  activation='relu'))
model.add(keras.layers.BatchNormalization(scale=False))
model.add(keras.layers.Dense(len(classes), activation='softmax'))

In [ ]:
lr = chosen_hps.get('learning_rate')
epochs = 500
batch_size = 64

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print(summary(model))

### Train

In [ ]:
history= model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=1)

### Evaluate

In [ ]:
result_all_models_FC, confusion_matrix_FC, predicitions_FC = evaluate_NN_models([model], X_test, y_test, bin_class=False)

In [ ]:
headers=['models','accuracy','precision','recall','f1_score']
data_normal_FC=[["model"]]
data_normal_FC.extend(result_all_models_FC)
print_table(data_normal_FC,headers)

In [ ]:
draw_confusion_matrix(confusion_matrix_FC,"Confusion Matrix")

In [ ]:
plt.figure()
# plt.plot(history.history['accuracy'], 'g', label='Training Accuracy')
plt.plot(history.history['val_accuracy'], color='orange', label='Validation Accuracy')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.show()

## Save

In [ ]:
# Convert our Keras model to a TFLite flatbuffer file
with open("z24.tflite", "wb") as flatbuffer_file:
    flatbuffer_bytes = lce.convert_keras_model(model)
    flatbuffer_file.write(flatbuffer_bytes)